# Final flagship notebook 05 — blocking verification

This is the flagship feature-based blocking notebook.

**Flagship defaults**
- uses `blocking_threshold = 0.1`
- writes to the flagship output tree
- ends with a compact readiness summary using truth-event counts and threshold-sweep diagnostics

In [1]:
from pathlib import Path
import sys
import json
from copy import deepcopy
import pandas as pd
from IPython.display import display, Markdown

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "src" / "flagship_predictability").exists() and (c / "examples" / "default_settings.py").exists():
            return c
    raise RuntimeError("Could not find bundle root with src/flagship_predictability and examples/default_settings.py")

BUNDLE_ROOT = detect_bundle_root()
if str(BUNDLE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT / "src"))
if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

FLAGSHIP_OUTPUT_ROOT = (BUNDLE_ROOT / "notebooks" / "outputs" / "flagship_final") if (BUNDLE_ROOT / "notebooks").exists() else (Path.cwd() / "outputs" / "flagship_final")
FLAGSHIP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from examples.default_settings import SETTINGS as BASE_SETTINGS

In [2]:
from flagship_predictability import run_blocking_verification

SETTINGS = deepcopy(BASE_SETTINGS)
SETTINGS.blocking_threshold = 0.1

display(Markdown("### Effective flagship settings"))
SETTINGS

### Effective flagship settings

AtlasConfig(truth_dataset='era5_truth_240', deterministic_models={'HRES': 'hres_0012_240', 'GraphCast': 'graphcast_2020_240', 'NeuralGCM': 'neuralgcm_det_2020_240'}, ensemble_models={'IFS-ENS': 'ifs_ens_240'}, date_windows=[('2020-01-01', '2020-03-31')], leads_hours=[24, 72, 120, 168], variables={'z500': VariableSpec(name='z500', candidates=['z', 'geopotential', 'gh'], level=500, climatology_groupby=None, thresholds=[]), 't850': VariableSpec(name='t850', candidates=['t', 'temperature'], level=850, climatology_groupby=None, thresholds=[]), 'u850': VariableSpec(name='u850', candidates=['u', 'u_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'v850': VariableSpec(name='v850', candidates=['v', 'v_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'mslp': VariableSpec(name='mslp', candidates=['msl', 'mean_sea_level_pressure', 'mslp'], level=None, climatology_groupby=None, thresholds=[])}, n_regimes=4, n_eof_components=8, bootstrap_n=400, boots

In [3]:
out = run_blocking_verification(SETTINGS, output_root=FLAGSHIP_OUTPUT_ROOT)
out

{'blocking_event_metrics': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_event_metrics.csv'),
 'blocking_rmse': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_rmse.csv'),
 'blocking_truth_fraction_summary': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_truth_fraction_summary.csv'),
 'blocking_threshold_sweep': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_threshold_sweep.csv'),
 'blocking_errors': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_errors.csv')}

In [4]:
from pathlib import Path

for key, path in out.items():
    path = Path(path)
    print(f"\n--- {key} ---")
    print(path)
    if path.suffix.lower() == ".csv" and path.exists():
        try:
            display(pd.read_csv(path).head(10))
        except Exception as e:
            print("Could not preview CSV:", e)


--- blocking_event_metrics ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_event_metrics.csv


,hits,misses,false_alarms,correct_negatives,POD,FAR,CSI,frequency_bias,truth_positive_count,forecast_positive_count,window,model,lead,sector
0,31,0,0,149,1.000000,0.000000,1.000000,1.000000,31,31,2020-01-01_2020-03-31,HRES,1 days,EuroAtlantic
1,29,0,2,149,1.000000,0.064516,0.935484,1.068966,29,31,2020-01-01_2020-03-31,HRES,1 days,Pacific
2,30,1,4,141,0.967742,0.117647,0.857143,1.096774,31,34,2020-01-01_2020-03-31,HRES,3 days,EuroAtlantic
3,26,3,2,145,0.896552,0.071429,0.838710,0.965517,29,28,2020-01-01_2020-03-31,HRES,3 days,Pacific
4,27,4,3,138,0.870968,0.100000,0.794118,0.967742,31,30,2020-01-01_2020-03-31,HRES,5 days,EuroAtlantic
5,21,8,7,136,0.724138,0.250000,0.583333,0.965517,29,28,2020-01-01_2020-03-31,HRES,5 days,Pacific
6,24,7,2,135,0.774194,0.076923,0.727273,0.838710,31,26,2020-01-01_2020-03-31,HRES,7 days,EuroAtlantic
7,20,9,7,132,0.689655,0.259259,0.555556,0.931034,29,27,2020-01-01_2020-03-31,HRES,7 days,Pacific
8,31,0,2,147,1.000000,0.060606,0.939394,1.064516,31,33,2020-01-01_2020-03-31,GraphCast,1 days,EuroAtlantic
9,29,0,1,150,1.000000,0.033333,0.966667,1.034483,29,30,2020-01-01_2020-03-31,GraphCast,1 days,Pacific



--- blocking_rmse ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_rmse.csv


,window,model,lead,blocked_rmse,unblocked_rmse,n_blocked,n_unblocked
0,2020-01-01_2020-03-31,HRES,1 days,49.530497,49.821034,53,127
1,2020-01-01_2020-03-31,HRES,3 days,140.356087,136.714731,53,123
2,2020-01-01_2020-03-31,HRES,5 days,316.847797,295.193281,53,119
3,2020-01-01_2020-03-31,HRES,7 days,538.754962,506.011876,53,115
4,2020-01-01_2020-03-31,GraphCast,1 days,39.725374,39.475633,53,127
5,2020-01-01_2020-03-31,GraphCast,3 days,122.893224,121.712517,53,123
6,2020-01-01_2020-03-31,GraphCast,5 days,271.856057,268.468279,53,119
7,2020-01-01_2020-03-31,GraphCast,7 days,485.591296,459.514497,53,115
8,2020-01-01_2020-03-31,NeuralGCM,1 days,37.716664,37.557958,53,127
9,2020-01-01_2020-03-31,NeuralGCM,3 days,114.142868,112.698997,53,123



--- blocking_truth_fraction_summary ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_truth_fraction_summary.csv


,window,model,lead,sector,truth_frac_mean,truth_frac_q90,truth_frac_q95,truth_frac_max,n_valid_times
0,2020-01-01_2020-03-31,HRES,1 days,EuroAtlantic,0.053194,0.18750,0.351250,0.475,180
1,2020-01-01_2020-03-31,HRES,1 days,Pacific,0.040556,0.13750,0.175625,0.500,180
2,2020-01-01_2020-03-31,HRES,3 days,EuroAtlantic,0.054403,0.18750,0.356250,0.475,176
3,2020-01-01_2020-03-31,HRES,3 days,Pacific,0.041477,0.13750,0.178125,0.500,176
4,2020-01-01_2020-03-31,HRES,5 days,EuroAtlantic,0.055451,0.18750,0.361250,0.475,172
5,2020-01-01_2020-03-31,HRES,5 days,Pacific,0.042442,0.13750,0.180625,0.500,172
6,2020-01-01_2020-03-31,HRES,7 days,EuroAtlantic,0.054688,0.19125,0.366250,0.475,168
7,2020-01-01_2020-03-31,HRES,7 days,Pacific,0.042783,0.14125,0.183125,0.500,168
8,2020-01-01_2020-03-31,GraphCast,1 days,EuroAtlantic,0.053194,0.18750,0.351250,0.475,180
9,2020-01-01_2020-03-31,GraphCast,1 days,Pacific,0.040556,0.13750,0.175625,0.500,180



--- blocking_threshold_sweep ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_threshold_sweep.csv


,window,model,lead,threshold,n_truth_blocked,fraction_truth_blocked
0,2020-01-01_2020-03-31,HRES,1 days,0.02,92,0.511111
1,2020-01-01_2020-03-31,HRES,1 days,0.05,78,0.433333
2,2020-01-01_2020-03-31,HRES,1 days,0.10,53,0.294444
3,2020-01-01_2020-03-31,HRES,1 days,0.20,23,0.127778
4,2020-01-01_2020-03-31,HRES,1 days,0.50,0,0.000000
5,2020-01-01_2020-03-31,HRES,3 days,0.02,92,0.522727
6,2020-01-01_2020-03-31,HRES,3 days,0.05,78,0.443182
7,2020-01-01_2020-03-31,HRES,3 days,0.10,53,0.301136
8,2020-01-01_2020-03-31,HRES,3 days,0.20,23,0.130682
9,2020-01-01_2020-03-31,HRES,3 days,0.50,0,0.000000



--- blocking_errors ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/blocking/blocking_errors.csv
Could not preview CSV: No columns to parse from file


In [5]:
events = pd.read_csv(out["blocking_event_metrics"])
rmse = pd.read_csv(out["blocking_rmse"])
truth_summary = pd.read_csv(out["blocking_truth_fraction_summary"])
thr = pd.read_csv(out["blocking_threshold_sweep"])

positive_events = events["truth_positive_count"].sum() if "truth_positive_count" in events.columns else 0
max_fraction = truth_summary["truth_frac_max"].max() if not truth_summary.empty else float("nan")
status = "PASS" if positive_events > 0 else "WARN"

display(Markdown("### Blocking readiness summary"))
summary = pd.DataFrame([
    {"metric": "total truth-positive sector events", "value": positive_events},
    {"metric": "max truth sector blocking fraction", "value": max_fraction},
    {"metric": "n blocked RMSE rows with finite blocked_rmse", "value": int(rmse["blocked_rmse"].notna().sum())},
])
display(summary)
display(Markdown(f"**Blocking flagship status:** {status}"))
display(Markdown("If this remains WARN, expand the window and/or keep the threshold sweep in the appendix."))

### Blocking readiness summary

,metric,value
0,total truth-positive sector events,720.0
1,max truth sector blocking fraction,0.5
2,n blocked RMSE rows with finite blocked_rmse,12.0


**Blocking flagship status:** PASS

If this remains WARN, expand the window and/or keep the threshold sweep in the appendix.